In [24]:
import random
import io
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from tensorflow import keras
from tensorflow.keras import layers
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding, Dropout
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.utils import pad_sequences 
from keras.layers import Input, TimeDistributed, CategoryEncoding, SimpleRNN, Dense
from keras.models import Model, Sequential

In [2]:
# descargar de textos.info
import urllib.request

# Para leer y parsear el texto en HTML de wikipedia
import bs4 as bs

In [33]:
raw_html = urllib.request.urlopen('https://www.textos.info/julio-verne/viaje-al-centro-de-la-tierra/ebook')
raw_html = raw_html.read()

# Parsear artículo, 'lxml' es el parser a utilizar
article_html = bs.BeautifulSoup(raw_html, 'lxml')

# Encontrar todos los párrafos del HTML (bajo el tag <p>)
# y tenerlos disponible como lista
article_paragraphs = article_html.find_all('p')

article_text = ''

for para in article_paragraphs:
    article_text += para.text + ' '

# pasar todo el texto a minúscula
article_text = article_text.lower()

In [34]:
article_text[:1000]

' el domingo 24 de mayo de 1863, mi tío, el profesor lidenbrock, entró \r\nrápidamente a su hogar, situado en el número 19 de la könig‑strasse, una\r\n de las calles más tradicionales del barrio antiguo de hamburgo. marta, su excelente criada, se preocupó sobremanera, creyendo que se \r\nhabía retrasado, pues apenas empezaba a cocinar la comida en el \r\nhornillo. “bueno” —pensé para mí—, “si mi tío viene con hambre, se va a armar \r\nla de san quintín; porque no conozco a otro hombre de menos paciencia”. —¡tan temprano y ya está aquí el señor lidenbrock! —exclamó la pobre marta, con arrebol, entreabriendo la puerta del comedor. —sí, marta; pero tú no tienes la culpa de que la comida no esté lista\r\n todavía, porque es temprano, aún no son las dos. acaba de dar la media \r\nhora en san miguel. —¿y por qué ha venido tan pronto el señor lidenbrock? —él lo explicará, seguramente. —¡ahí viene! yo me escapo. señor axel, cálmelo usted, por favor. y la excelente marta se retiró presurosa a s

In [35]:
max_context_size = 100

In [36]:
chars_vocab = set(article_text)

In [37]:
char2idx = {k: v for v,k in enumerate(chars_vocab)}
idx2char = {v: k for k,v in char2idx.items()}

In [38]:
tokenized_text = [char2idx[ch] for ch in article_text]

In [39]:
tokenized_text[:1000]

[44,
 58,
 49,
 44,
 43,
 63,
 2,
 53,
 31,
 16,
 63,
 44,
 34,
 51,
 44,
 43,
 58,
 44,
 2,
 36,
 13,
 63,
 44,
 43,
 58,
 44,
 55,
 42,
 27,
 77,
 66,
 44,
 2,
 53,
 44,
 40,
 32,
 63,
 66,
 44,
 58,
 49,
 44,
 59,
 48,
 63,
 4,
 58,
 71,
 63,
 48,
 44,
 49,
 53,
 43,
 58,
 31,
 22,
 48,
 63,
 74,
 67,
 66,
 44,
 58,
 31,
 40,
 48,
 76,
 44,
 46,
 28,
 48,
 75,
 59,
 53,
 43,
 36,
 2,
 58,
 31,
 40,
 58,
 44,
 36,
 44,
 71,
 5,
 44,
 64,
 63,
 16,
 36,
 48,
 66,
 44,
 71,
 53,
 40,
 5,
 36,
 43,
 63,
 44,
 58,
 31,
 44,
 58,
 49,
 44,
 31,
 17,
 2,
 58,
 48,
 63,
 44,
 55,
 9,
 44,
 43,
 58,
 44,
 49,
 36,
 44,
 67,
 47,
 31,
 53,
 16,
 15,
 71,
 40,
 48,
 36,
 71,
 71,
 58,
 66,
 44,
 5,
 31,
 36,
 46,
 28,
 44,
 43,
 58,
 44,
 49,
 36,
 71,
 44,
 74,
 36,
 49,
 49,
 58,
 71,
 44,
 2,
 75,
 71,
 44,
 40,
 48,
 36,
 43,
 53,
 74,
 53,
 63,
 31,
 36,
 49,
 58,
 71,
 44,
 43,
 58,
 49,
 44,
 22,
 36,
 48,
 48,
 53,
 63,
 44,
 36,
 31,
 40,
 53,
 16,
 5,
 63,
 44,
 43,
 58,
 44,
 64,
 3

In [40]:
p_val = 0.1
num_val = int(np.ceil(len(tokenized_text)*p_val/max_context_size))

In [41]:
train_text = tokenized_text[:-num_val*max_context_size]
val_text = tokenized_text[-num_val*max_context_size:]

In [42]:
tokenized_sentences_val = [val_text[init*max_context_size:init*(max_context_size+1)] for init in range(num_val)]

In [43]:
tokenized_sentences_train = [train_text[init:init+max_context_size] for init in range(len(train_text)-max_context_size+1)]

In [44]:
X = np.array(tokenized_sentences_train[:-1])
y = np.array(tokenized_sentences_train[1:])

In [45]:
vocab_size = len(chars_vocab)

Modelo base

In [46]:
model = Sequential()

model.add(TimeDistributed(CategoryEncoding(num_tokens=vocab_size, output_mode = "one_hot"),input_shape=(None,1)))
model.add(SimpleRNN(200, return_sequences=True, dropout=0.1, recurrent_dropout=0.1 ))
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='sparse_categorical_crossentropy', optimizer='rmsprop')

model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 time_distributed_1 (TimeDi  (None, None, 78)          0         
 stributed)                                                      
                                                                 
 simple_rnn_1 (SimpleRNN)    (None, None, 200)         55800     
                                                                 
 dense_1 (Dense)             (None, None, 78)          15678     
                                                                 
Total params: 71478 (279.21 KB)
Trainable params: 71478 (279.21 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [47]:
class PplCallback(keras.callbacks.Callback):

    '''
    Este callback es una solución ad-hoc para calcular al final de cada epoch de
    entrenamiento la métrica de Perplejidad sobre un conjunto de datos de validación.
    La perplejidad es una métrica cuantitativa para evaluar la calidad de la generación de secuencias.
    Además implementa la finalización del entrenamiento (Early Stopping)
    si la perplejidad no mejora después de `patience` epochs.
    '''

    def __init__(self, val_data, history_ppl,patience=5):
      # El callback lo inicializamos con secuencias de validación sobre las cuales
      # mediremos la perplejidad
      self.val_data = val_data

      self.target = []
      self.padded = []

      count = 0
      self.info = []
      self.min_score = np.inf
      self.patience_counter = 0
      self.patience = patience

      # nos movemos en todas las secuencias de los datos de validación
      for seq in self.val_data:

        len_seq = len(seq)
        # armamos todas las subsecuencias
        subseq = [seq[:i] for i in range(1,len_seq)]
        self.target.extend([seq[i] for i in range(1,len_seq)])

        if len(subseq)!=0:

          self.padded.append(pad_sequences(subseq, maxlen=max_context_size, padding='pre'))

          self.info.append((count,count+len_seq))
          count += len_seq

      self.padded = np.vstack(self.padded)


    def on_epoch_end(self, epoch, logs=None):

        # en `scores` iremos guardando la perplejidad de cada secuencia
        scores = []

        predictions = self.model.predict(self.padded,verbose=0)

        # para cada secuencia de validación
        for start,end in self.info:

          # en `probs` iremos guardando las probabilidades de los términos target
          probs = [predictions[idx_seq,-1,idx_vocab] for idx_seq, idx_vocab in zip(range(start,end),self.target[start:end])]

          # calculamos la perplejidad por medio de logaritmos
          scores.append(np.exp(-np.sum(np.log(probs))/(end-start)))

        # promediamos todos los scores e imprimimos el valor promedio
        current_score = np.mean(scores)
        history_ppl.append(current_score)
        print(f'\n mean perplexity: {current_score} \n')

        # chequeamos si tenemos que detener el entrenamiento
        if current_score < self.min_score:
          self.min_score = current_score
          self.model.save("my_model.keras")
          print("Saved new model!")
          self.patience_counter = 0
        else:
          self.patience_counter += 1
          if self.patience_counter == self.patience:
            print("Stopping training...")
            self.model.stop_training = True

Entrenamiento

In [48]:
history_ppl = []
hist = model.fit(X, y, epochs=20, callbacks=[PplCallback(tokenized_sentences_val,history_ppl)], batch_size=256)

Epoch 1/20
1522/1522 [==============================] - ETA: 0s - loss: 2.2067

MemoryError: Unable to allocate 2.68 GiB for an array with shape (92374, 100, 78) and data type float32